### Primer modelo sin usar framework 

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd


In [2]:
# Ruta de la carpeta donde se guardaron los datos
RUTA_DATOS_MODELO = Path("../datos_modelo")


# Cargamos las características
X_train = np.load(
    RUTA_DATOS_MODELO / "X_train.npy",
    allow_pickle=False
)

X_test = np.load(
    RUTA_DATOS_MODELO / "X_test.npy",
    allow_pickle=False
)


# Cargamos las etiquetas
y_train = np.load(
    RUTA_DATOS_MODELO / "y_train.npy",
    allow_pickle=False
)

y_test = np.load(
    RUTA_DATOS_MODELO / "y_test.npy",
    allow_pickle=False
)


# Cargamos los nombres de las características
nombres_caracteristicas = np.load(
    RUTA_DATOS_MODELO / "nombres_caracteristicas.npy",
    allow_pickle=True
)


# Mostramos las dimensiones para comprobar la carga
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print(
    "Nombres de características:",
    nombres_caracteristicas.shape
)

X_train: (12448, 120)
y_train: (12448,)
X_test: (3112, 120)
y_test: (3112,)
Nombres de características: (120,)


## 1. Estandarización de las características

KNN clasifica una observación según la distancia que existe entre sus características y las observaciones de entrenamiento. Por esta razón, las características deben encontrarse en escalas comparables.

La media y la desviación estándar se calcularán únicamente con `X_train`. Después, esos mismos valores se utilizarán para transformar `X_train` y `X_test`. Esto evita utilizar información del conjunto de prueba durante la preparación del modelo.

In [3]:
# ---------------------------------------------------------
# Cálculo de los parámetros con entrenamiento
# ---------------------------------------------------------

# Media de cada una de las 120 características
media_train = np.mean(
    X_train,
    axis=0
)

# Desviación estándar de cada característica
desviacion_train = np.std(
    X_train,
    axis=0
)


# Algunas características podrían tener desviación igual a
# cero. En esos casos se utiliza 1 para evitar una división
# entre cero. La característica permanecerá con valor cero.
desviacion_train[desviacion_train == 0] = 1


# ---------------------------------------------------------
# Aplicación de la estandarización
# ---------------------------------------------------------

X_train_escalado = (
    X_train - media_train
) / desviacion_train

X_test_escalado = (
    X_test - media_train
) / desviacion_train


# Se utiliza float32 para reducir el uso de memoria durante
# el cálculo de las distancias.
X_train_escalado = X_train_escalado.astype(
    np.float32
)

X_test_escalado = X_test_escalado.astype(
    np.float32
)


print(
    "X_train escalado:",
    X_train_escalado.shape
)

print(
    "X_test escalado:",
    X_test_escalado.shape
)

print(
    "Valores faltantes en entrenamiento:",
    np.isnan(X_train_escalado).sum()
)

print(
    "Valores faltantes en prueba:",
    np.isnan(X_test_escalado).sum()
)

X_train escalado: (12448, 120)
X_test escalado: (3112, 120)
Valores faltantes en entrenamiento: 0
Valores faltantes en prueba: 0


### Interpretación

Las características de entrenamiento y prueba conservaron sus dimensiones originales, pero ahora se encuentran en escalas comparables. La transformación utilizó exclusivamente la media y desviación estándar del conjunto de entrenamiento.

También se verificó que la estandarización no generara valores faltantes. Los datos escalados ya pueden utilizarse para calcular las distancias del algoritmo KNN.

## 2. Implementación de KNN desde cero

K-Nearest Neighbors clasifica una nueva ventana buscando las observaciones de entrenamiento más cercanas. La actividad más frecuente entre los `k` vecinos seleccionados se utiliza como predicción.

La implementación seguirá estos pasos:

1. Guardar las características y etiquetas de entrenamiento.
2. Calcular la distancia euclidiana entre cada ventana de prueba y las ventanas de entrenamiento.
3. Seleccionar los `k` vecinos con menor distancia.
4. Contar las etiquetas de los vecinos.
5. Asignar la actividad con mayor cantidad de votos.

Las distancias se calcularán por bloques para evitar almacenar simultáneamente una matriz demasiado grande en memoria.

In [4]:
# ---------------------------------------------------------
# Implementación manual del clasificador KNN
# ---------------------------------------------------------

class KNNDesdeCero:
    """
    Clasificador K-Nearest Neighbors implementado con NumPy.
    """

    def __init__(self, k=5, tamanio_bloque=64):
        """
        k:
            Cantidad de vecinos utilizados para clasificar.

        tamanio_bloque:
            Cantidad de muestras de prueba procesadas
            simultáneamente para controlar el uso de memoria.
        """
        self.k = k
        self.tamanio_bloque = tamanio_bloque


    def fit(self, X, y):
        """
        Guarda los datos de entrenamiento.

        KNN no construye una ecuación durante fit, porque
        necesita conservar las observaciones para compararlas
        posteriormente con las muestras nuevas.
        """

        if len(X) != len(y):
            raise ValueError(
                "X y y deben tener la misma cantidad de filas."
            )

        if self.k < 1 or self.k > len(X):
            raise ValueError(
                "k debe estar entre 1 y la cantidad de "
                "muestras de entrenamiento."
            )

        self.X_train = np.asarray(
            X,
            dtype=np.float32
        )

        self.y_train = np.asarray(y)

        # Clases disponibles y representación numérica de
        # cada etiqueta. Esto facilita el conteo de votos.
        self.clases, self.y_numerico = np.unique(
            self.y_train,
            return_inverse=True
        )

        return self


    def _predecir_bloque(self, X_bloque):
        """
        Predice un bloque de muestras para controlar el
        consumo de memoria.
        """

        # Distancia euclidiana al cuadrado:
        #
        # ||a-b||² = ||a||² + ||b||² - 2(a·b)
        #
        # No es necesario calcular la raíz cuadrada porque
        # no cambia el orden de las distancias.

        norma_prueba = np.sum(
            X_bloque ** 2,
            axis=1,
            keepdims=True
        )

        norma_train = np.sum(
            self.X_train ** 2,
            axis=1
        )

        distancias = (
            norma_prueba
            + norma_train
            - 2 * X_bloque @ self.X_train.T
        )

        # Corregimos posibles valores negativos muy pequeños
        # generados por el redondeo numérico.
        distancias = np.maximum(
            distancias,
            0
        )

        # Localizamos los índices de los k vecinos con menor
        # distancia para cada muestra del bloque.
        indices_vecinos = np.argpartition(
            distancias,
            kth=self.k - 1,
            axis=1
        )[:, :self.k]

        predicciones_bloque = []

        # Realizamos la votación para cada muestra.
        for fila, vecinos in enumerate(indices_vecinos):

            etiquetas_vecinos = self.y_numerico[
                vecinos
            ]

            votos = np.bincount(
                etiquetas_vecinos,
                minlength=len(self.clases)
            )

            clases_ganadoras = np.flatnonzero(
                votos == votos.max()
            )

            # Normalmente habrá una sola clase ganadora.
            if len(clases_ganadoras) == 1:
                clase_elegida = clases_ganadoras[0]

            else:
                # Si hay empate, se elige entre las clases
                # empatadas aquella cuyo vecino esté más
                # cerca de la muestra evaluada.
                distancias_vecinos = distancias[
                    fila,
                    vecinos
                ]

                orden = np.argsort(
                    distancias_vecinos
                )

                clase_elegida = None

                for posicion in orden:
                    clase_vecino = etiquetas_vecinos[
                        posicion
                    ]

                    if clase_vecino in clases_ganadoras:
                        clase_elegida = clase_vecino
                        break

            predicciones_bloque.append(
                self.clases[clase_elegida]
            )

        return np.asarray(predicciones_bloque)


    def predict(self, X):
        """
        Predice las etiquetas procesando los datos por
        bloques.
        """

        X = np.asarray(
            X,
            dtype=np.float32
        )

        predicciones = []

        # Procesamos el conjunto de prueba en bloques.
        for inicio in range(
            0,
            len(X),
            self.tamanio_bloque
        ):

            fin = inicio + self.tamanio_bloque

            X_bloque = X[inicio:fin]

            predicciones_bloque = (
                self._predecir_bloque(X_bloque)
            )

            predicciones.extend(
                predicciones_bloque
            )

        return np.asarray(predicciones)

## 3. Entrenamiento y evaluación del modelo

Se utilizarán cinco vecinos para realizar cada clasificación. El método `fit()` almacenará las ventanas de entrenamiento y sus etiquetas. Posteriormente, `predict()` calculará las distancias y generará una predicción para cada ventana de prueba.

Finalmente, las predicciones se compararán con las etiquetas reales para calcular la exactitud del modelo.

In [5]:
# ---------------------------------------------------------
# Creación y entrenamiento del modelo
# ---------------------------------------------------------

modelo_knn = KNNDesdeCero(
    k=5,
    tamanio_bloque=64
)

modelo_knn.fit(
    X_train_escalado,
    y_train
)

print("Datos de entrenamiento almacenados.")


# ---------------------------------------------------------
# Predicción de las ventanas de prueba
# ---------------------------------------------------------

# Este paso puede tardar porque cada ventana de prueba se
# compara con las 12,448 ventanas de entrenamiento.
y_pred_knn = modelo_knn.predict(
    X_test_escalado
)

print(
    "Cantidad de predicciones:",
    len(y_pred_knn)
)


# ---------------------------------------------------------
# Cálculo manual de la exactitud
# ---------------------------------------------------------

predicciones_correctas = np.sum(
    y_pred_knn == y_test
)

exactitud_knn = (
    predicciones_correctas / len(y_test)
)

print(
    "Predicciones correctas:",
    predicciones_correctas
)

print(
    f"Exactitud por ventana: {exactitud_knn:.4f}"
)

print(
    f"Porcentaje de aciertos: "
    f"{exactitud_knn * 100:.2f}%"
)

Datos de entrenamiento almacenados.
Cantidad de predicciones: 3112
Predicciones correctas: 2763
Exactitud por ventana: 0.8879
Porcentaje de aciertos: 88.79%


### Interpretación

El modelo KNN implementado desde cero clasificó correctamente **2,763 de las 3,112 ventanas** del conjunto de prueba, obteniendo una exactitud de **88.79%**. Esto significa que aproximadamente 89 de cada 100 ventanas fueron asignadas a la actividad correcta.

El resultado muestra que las características estadísticas extraídas contienen información útil para distinguir las actividades, incluso utilizando un algoritmo basado únicamente en la cercanía entre observaciones.

Este resultado corresponde a una primera configuración con `k=5`. Todavía no se han comparado otros valores de `k` ni se ha realizado un ajuste de hiperparámetros. Además, la evaluación se realizó por ventana; posteriormente se pueden combinar las cuatro predicciones de cada señal para obtener una sola actividad por señal completa.

## 4. Predicción por señal completa

El modelo KNN genera una predicción para cada ventana de 220 puntos. Como cada señal original fue dividida en cuatro ventanas, se combinarán las cuatro predicciones mediante votación mayoritaria para obtener una sola actividad por señal completa.

Los metadatos se utilizarán únicamente para agrupar las ventanas que tienen el mismo `sample_id`. Estos datos no participan como características del modelo.

In [8]:
# ---------------------------------------------------------
# Carga de los metadatos de prueba
# ---------------------------------------------------------

metadatos_test = pd.read_csv(
    RUTA_DATOS_MODELO / "metadatos_test.csv",
    dtype={
        "sample_id": str,
        "actividad": str
    }
)


# Verificamos que cada predicción tenga su metadato.
if len(metadatos_test) != len(y_pred_knn):
    raise ValueError(
        "La cantidad de predicciones no coincide con "
        "la cantidad de metadatos."
    )


# ---------------------------------------------------------
# Tabla con los resultados de cada ventana
# ---------------------------------------------------------

resultados_ventanas_knn = metadatos_test.copy()

resultados_ventanas_knn["actividad_real"] = y_test
resultados_ventanas_knn["actividad_predicha"] = y_pred_knn

In [9]:
# ---------------------------------------------------------
# Votación mayoritaria por señal completa
# ---------------------------------------------------------

resultados_senales_knn = []

# Agrupamos las cuatro ventanas de cada señal.
for sample_id, grupo in resultados_ventanas_knn.groupby(
    "sample_id",
    sort=False
):
    # Las cuatro ventanas tienen la misma actividad real.
    actividad_real = grupo[
        "actividad_real"
    ].iloc[0]

    # Contamos cuántas veces fue predicha cada actividad.
    conteo_votos = grupo[
        "actividad_predicha"
    ].value_counts()

    # Seleccionamos la actividad con más votos.
    actividad_predicha = conteo_votos.index[0]

    resultados_senales_knn.append({
        "sample_id": sample_id,
        "actividad_real": actividad_real,
        "actividad_predicha": actividad_predicha,
        "cantidad_ventanas": len(grupo)
    })


# Convertimos los resultados en una tabla.
resultados_senales_knn = pd.DataFrame(
    resultados_senales_knn
)


display(
    resultados_senales_knn.head()
)

print(
    "Cantidad de señales completas:",
    len(resultados_senales_knn)
)

print(
    "Ventanas por señal:",
    resultados_senales_knn[
        "cantidad_ventanas"
    ].unique()
)

,sample_id,actividad_real,actividad_predicha,cantidad_ventanas
0,009_0168,009,006,4
1,008_0019,008,008,4
2,004_0260,004,004,4
3,004_0050,004,004,4
4,015_0069,015,015,4


Cantidad de señales completas: 778
Ventanas por señal: [4]


In [10]:
# ---------------------------------------------------------
# Obtención de las etiquetas finales
# ---------------------------------------------------------

y_test_senales_knn = resultados_senales_knn[
    "actividad_real"
].to_numpy()

y_pred_senales_knn = resultados_senales_knn[
    "actividad_predicha"
].to_numpy()


# ---------------------------------------------------------
# Cálculo manual de la exactitud
# ---------------------------------------------------------

aciertos_senales_knn = np.sum(
    y_test_senales_knn == y_pred_senales_knn
)

exactitud_senales_knn = (
    aciertos_senales_knn
    / len(y_test_senales_knn)
)


print(
    "Señales clasificadas correctamente:",
    aciertos_senales_knn
)

print(
    "Cantidad total de señales:",
    len(y_test_senales_knn)
)

print(
    f"Exactitud por señal completa: "
    f"{exactitud_senales_knn:.4f}"
)

print(
    f"Porcentaje de aciertos: "
    f"{exactitud_senales_knn * 100:.2f}%"
)

Señales clasificadas correctamente: 728
Cantidad total de señales: 778
Exactitud por señal completa: 0.9357
Porcentaje de aciertos: 93.57%


### Interpretación

Después de combinar mediante votación mayoritaria las predicciones de las cuatro ventanas, el modelo KNN clasificó correctamente **728 de las 778 señales completas** del conjunto de prueba. Esto corresponde a una exactitud de **93.57%**.

El resultado por señal completa fue mayor que el obtenido por ventana, que fue de **88.79%**. Esto indica que algunos errores ocurrieron únicamente en una parte de la señal, mientras que las demás ventanas permitieron recuperar la actividad correcta mediante la votación mayoritaria.

Por lo tanto, dividir las señales en ventanas permitió analizar diferentes fragmentos del movimiento y posteriormente combinar sus resultados para obtener una clasificación más estable de la actividad completa.

Este resultado corresponde a la configuración inicial de KNN con `k=5`. Todavía no se han probado otros valores de `k` ni se ha realizado un ajuste de hiperparámetros. En caso de empate entre las predicciones de las cuatro ventanas, el procedimiento selecciona la primera actividad entre las que obtuvieron la mayor cantidad de votos.